# Compare expression predictions

Run notebook 02 first. Evaluate zero-shot and fine-tuned Impleo using all cells and identical target genes. External baseline predictions can be supplied as aligned NPZ files.

In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "examples":
    ROOT = ROOT.parent
if not (ROOT / "impleo").is_dir():
    raise RuntimeError("Open this notebook from the Impleo repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ASSETS = ROOT / "data" / "assets"
EXAMPLE = ASSETS / "starmap_p128_v0"
if not (EXAMPLE / "input_panel.npy").exists():
    subprocess.run([sys.executable, "-m", "data.download", "--packs", "example",
                    "--out", str(ASSETS)], cwd=ROOT, check=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(6)
from impleo.inference import read_names
panel = read_names(EXAMPLE / "protocol.json", "panel_genes")
targets = read_names(EXAMPLE / "hidden_genes.json")
values = np.load(EXAMPLE / "input_panel.npy", allow_pickle=False)
truth = np.load(EXAMPLE / "truth.npy", allow_pickle=False)
CHECKPOINT = ROOT / "checkpoints" / "impleo.pt"
print(f"{len(values)} cells; {len(panel)} panel genes; {len(targets)} target genes; {DEVICE}")

In [ ]:
from evaluation.benchmark import run
from evaluation.plotting import draw

# Add independently generated baseline NPZ files here (see README).
external = {}
# external["SpaGE"] = ROOT / "results" / "spage.npz"
# external["Tangram"] = ROOT / "results" / "tangram.npz"
# external["scGPT-spatial"] = ROOT / "results" / "scgpt_spatial.npz"

cell_ids = np.arange(len(values)).astype(str)
FT_CHECKPOINT = ROOT / "results" / "finetuning" / "best.pt"
if not FT_CHECKPOINT.exists():
    raise FileNotFoundError("Run notebook 02_finetuning first")
scores, summary = run(CHECKPOINT, values, panel, targets, truth, cell_ids,
                      ROOT / "results" / "benchmark",
                      finetuned_checkpoint=FT_CHECKPOINT,
                      external=external, device=DEVICE)
summary

In [ ]:
draw(scores, ROOT / "results" / "benchmark" / "figures")
from IPython.display import Image, display
display(Image(filename=str(ROOT / "results" / "benchmark" / "figures" / "barplot.png")))